# 04 — Topological order and full backpropagation

Part of the micrograd repetition pack.


## Goal
Traverse an arbitrary computation graph so every node receives its gradient only after downstream gradients are available.


In [ ]:
import math

_results = []

def check(name, condition, detail=""):
    ok = bool(condition)
    _results.append(ok)
    mark = "PASS" if ok else "FAIL"
    print(f"[{mark}] {name}" + (f" — {detail}" if detail else ""))

def close(a, b, tol=1e-6):
    return abs(a - b) <= tol

def summary():
    print(f"\nScore: {sum(_results)}/{len(_results)} tests passed")


In [ ]:
from math import exp, log, sin, cos

class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def exp(self):
        out = Value(exp(self.data), (self,), 'exp')
        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward
        return out

    def log(self):
        out = Value(log(self.data), (self,), 'log')
        def _backward():
            self.grad += (1 / self.data) * out.grad
        out._backward = _backward
        return out

    def __pow__(self, power):
        assert isinstance(power, (int, float))
        out = Value(self.data ** power, (self,), f'**{power}')
        def _backward():
            self.grad += power * self.data ** (power - 1) * out.grad
        out._backward = _backward
        return out

    def sin(self):
        out = Value(sin(self.data), (self,), 'sin')
        def _backward():
            self.grad += cos(self.data) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        e2x = (2 * self).exp()
        return (e2x - 1) / (e2x + 1)

    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return Value(other) + (-self)
    def __radd__(self, other): return self + other
    def __rmul__(self, other): return self * other
    def __truediv__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return self * other ** -1
    def __rtruediv__(self, other): return Value(other) * self ** -1

    def backward_reference(self):
        topo, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)
        build(self)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()


### Round A — Trace a graph
Return all nodes and directed `(child, parent)` edges.


In [ ]:
def trace(root):
    """TODO: recursively collect nodes and edges without duplicates."""
    raise NotImplementedError

_results.clear()
try:
    a,b=Value(2),Value(3); c=a*b+a
    nodes,edges=trace(c)
    check("four graph nodes", len(nodes)==4)
    check("four directed edges", len(edges)==4)
    check("root included", c in nodes)
except Exception as e: check("trace runs", False, repr(e))
summary()


### Round B — Topological order
Return children before parents. A shared node must appear only once.


In [ ]:
def topological(root):
    """TODO: depth-first traversal with a visited set."""
    raise NotImplementedError

_results.clear()
try:
    a,b=Value(2),Value(3); q=a*b; out=q+a
    topo=topological(out)
    check("every node once", len(topo)==len(set(topo))==4)
    check("root last", topo[-1] is out)
    check("children before product", topo.index(a)<topo.index(q) and topo.index(b)<topo.index(q))
except Exception as e: check("topological runs", False, repr(e))
summary()


### Round C — Implement `backward(root)`
Seed the output gradient with 1, then run node `_backward` functions in reverse topological order.


In [ ]:
def backward(root):
    """TODO."""
    raise NotImplementedError

_results.clear()
try:
    x,y=Value(2),Value(3); z=x*y+x
    backward(z)
    check("dz/dx", x.grad==4)
    check("dz/dy", y.grad==2)
    x=Value(3); z=x*x+x
    backward(z)
    check("shared-node accumulation", x.grad==7)
except Exception as e: check("backward runs", False, repr(e))
summary()


### Round D — Original expression
Construct the source notebook expression entirely with `Value` operations, run your `backward`, and match the analytical gradient.


In [ ]:
_results.clear()
try:
    a,b,c=Value(2),Value(3),Value(4)
    L=-a**3+(3*b).sin()-1/c+b**2.5-a**0.5
    backward(L)
    expected=[-12.353553390593273,10.25699027111255,0.0625]
    for name,v,e in zip("abc",[a,b,c],expected): check(f"dL/d{name}",close(v.grad,e))
except Exception as e: check("original expression",False,repr(e))
summary()


### Concept checks

1. Why is the forward topological list reversed for backpropagation?
2. What breaks if a parent runs before all of its children have contributed gradients?
3. Why is a visited set required for `x*x+x`?
4. Rebuild `topological` and `backward` without notes.
